# Configuracion y Carga de Datos

In [1]:
import pandas as pd
import numpy as np
import os

from quant_utils import DataDownloader, DataManager, FinancialStats, PortfolioOptimizer
from quant_utils.brokers import BrokerFactory
from quant_utils.reports.plotting import QuantPlotter
from quant_utils.reports.report_generator import ReportGenerator


import yfinance as yf

In [2]:
activos = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META"]
ticker_benchmark = "QQQ"

start_date = "2021-01-01"
end_date = "2025-12-31"
tasa_libre_riesgo_anual = 0.04


split_date = pd.to_datetime("2025-01-01").date() # Fecha de corte Entrenamiento / Validación

# Descarga y cálculo de retornos idéntico al anterior
DataDownloader.download_portfolio_data(tickers=activos, start_date=start_date, end_date=end_date, filename="precios_historicos.parquet")
precios = DataManager.load_parquet("raw", "precios_historicos.parquet")
retornos_totales = FinancialStats.calcular_retornos_log(precios)


# ==============================================================================
# PARÁMETROS DE CONFIGURACIÓN DEL BENCHMARK
# ==============================================================================
carpeta_cache = os.path.join("data","raw")
archivo_cache = os.path.join(carpeta_cache, f"benchmark_{ticker_benchmark}.csv")

# Creamos la carpeta de caché si no existe
os.makedirs(carpeta_cache, exist_ok=True)

# ==============================================================================
# SISTEMA DE CACHÉ INTELIGENTE (EVITA DESCARGAS REPETIDAS)
# ==============================================================================
descargar_de_nuevo = True

# Convertimos los límites a Timestamps de Pandas para comparar directamente
ts_split = pd.to_datetime(split_date)
ts_end = pd.to_datetime(end_date)

if os.path.exists(archivo_cache):
    # Leemos el archivo CSV guardado localmente
    df_local = pd.read_csv(archivo_cache, index_col=0)
    df_local.index = pd.to_datetime(df_local.index, errors='coerce')
    
    # Limpiamos las filas que no tengan una fecha válida en el índice
    df_local = df_local[df_local.index.notna()]
    
    if not df_local.empty:
        # Comparamos Timestamps directamente (sin usar .date(), lo que evita el AttributeError)
        if df_local.index.min() <= ts_split and df_local.index.max() >= ts_end:
            print(f"⚡ [CACHÉ] Cargando {ticker_benchmark} desde el disco local...")
            
            # Buscamos la columna de precio de forma flexible
            columna_cierre = [c for c in df_local.columns if 'adj' in str(c).lower() or 'close' in str(c).lower()][0]
            df_bench = df_local[columna_cierre]
            descargar_de_nuevo = False

if descargar_de_nuevo:
    print(f"📥 [YFINANCE] Descargando {ticker_benchmark} desde Yahoo Finance...")
    # Forzamos la descarga limpia sin MultiIndex complejo
    df_descarga = yf.download(ticker_benchmark, start=split_date, end=end_date, auto_adjust=False)
    # Guardamos el CSV para la caché
    df_descarga.to_csv(archivo_cache)
    
    # Buscamos la columna dinámicamente ignorando mayúsculas/minúsculas o espacios
    columnas_candidatas = [c for c in df_descarga.columns if 'adj' in str(c).lower()]
    if not columnas_candidatas: # Si auto-adjust se activó solo
        columnas_candidatas = [c for c in df_descarga.columns if 'close' in str(c).lower()]
        
    col_interes = columnas_candidatas[0]
    df_bench = df_descarga[col_interes]


df_bench = pd.Series(df_bench.values.flatten(), index=df_bench.index)
# Calculamos los retornos logarítmicos diarios del Benchmark para que hablen el mismo idioma
retornos_log_bench = FinancialStats.calcular_retornos_log(df_bench)
curva_benchmark = np.exp(retornos_log_bench.cumsum()).dropna()

# split exacto de las matrices de retornos logarítmicos
retornos_is = retornos_totales.loc[:split_date]   # IN-SAMPLE (Entrenamiento)
retornos_oos = retornos_totales.loc[split_date:]  # OUT-OF-SAMPLE (Validación)

print(f"📁 In-Sample: {retornos_is.shape[0]} días | Out-of-Sample: {retornos_oos.shape[0]} días.")

- Iniciando descarga de: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META']
- Período: 2021-01-01 al 2025-12-31


[*********************100%***********************]  6 of 6 completed
/tmp/ipykernel_6641/4063668309.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_local.index = pd.to_datetime(df_local.index, errors='coerce')


¡Datos guardados exitosamente en: /home/juanm/dev/Quant/quant-labs/data/raw/precios_historicos.parquet!
Registros: 1254 días de mercado | 6 activos.
📥 [YFINANCE] Descargando QQQ desde Yahoo Finance...


[*********************100%***********************]  1 of 1 completed

📁 In-Sample: 1004 días | Out-of-Sample: 249 días.


# Simulación de Estrategias y Alocación

In [3]:
pesos_markowitz_estatico = PortfolioOptimizer.get_max_sharpe(retornos_is,max_peso=0.35, tasa_libre_riesgo=0.0, gamma=0.0)
pesos_markowitz_regularizado = PortfolioOptimizer.get_max_sharpe(retornos_is,max_peso=0.35, tasa_libre_riesgo=0.0, gamma=0.02)

print(f"🎯 Pesos Markowitz Puro entrenados (IS): {np.round(pesos_markowitz_estatico, 3)}")
print(f"🛡️ Pesos Markowitz Regularizado (IS):  {np.round(pesos_markowitz_regularizado, 3)}")

# Inicializamos pesos pasivos de control
pesos_1n = np.ones(len(activos)) / len(activos)

# Ejecutamos el producto punto de los vectores fijos entrenados contra los datos NUEVOS (OOS)
ret_log_oos_1n = retornos_oos.dot(pesos_1n)
ret_log_oos_estatico = retornos_oos.dot(pesos_markowitz_estatico)
ret_log_oos_regularizado = retornos_oos.dot(pesos_markowitz_regularizado)

# Reconstrucción real de las Curvas de Equidad (Capitalización Continua)
curva_oos_1n = np.exp(ret_log_oos_1n.cumsum()).dropna()
curva_oos_estatico = np.exp(ret_log_oos_estatico.cumsum()).dropna()
curva_oos_regularizado = np.exp(ret_log_oos_regularizado.cumsum()).dropna()

print("📈 Curvas de equidad Out-of-Sample generadas bajo estricta separación de entornos.")

🎯 Pesos Markowitz Puro entrenados (IS): [0.291 0.    0.232 0.019 0.109 0.35 ]
🛡️ Pesos Markowitz Regularizado (IS):  [0.251 0.    0.193 0.039 0.167 0.35 ]
📈 Curvas de equidad Out-of-Sample generadas bajo estricta separación de entornos.


# Auditoria de Fricción y Costos Operativos

In [4]:
capital_inicial = 10000.0
lista_pesos_dinamicos = []

# 1. Generamos las fechas de rebalanceo (Fin de cada mes) dentro del periodo Out-of-Sample
fechas_rebalanceo = pd.date_range(start=split_date, end=end_date, freq='ME').date

# Inicializamos el vector de pesos previos (1/N) para la primera penalización de turnover
pesos_previos = np.ones(len(activos)) / len(activos)

print("⏳ Ejecutando optimizaciones mensuales Walk-Forward en periodo de validación...")

# 2. Bucle de re-optimización periódica
for fecha in fechas_rebalanceo:
    # Ventana móvil hacia atrás: tomamos los últimos 252 días de mercado desde la fecha de corte
    datos_ventana = retornos_totales.loc[:fecha].tail(252)
    
    # Invocamos al optimizador con las restricciones institucionales corregidas
    pesos_mes = PortfolioOptimizer.get_max_sharpe(
        retornos=datos_ventana, 
        pesos_previos=pesos_previos, 
        tasa_libre_riesgo=0.04,
        max_peso=0.35,  # 🛡️ Forzamos diversificación máxima por activo (Adiós concentración en META)
        gamma=0.02      # 📉 Penalizamos cambios bruscos para mitigar comisiones del broker
    )
    
    lista_pesos_dinamicos.append(pesos_mes)
    pesos_previos = pesos_mes  # Actualizamos el estado para el próximo mes

# 3. Construimos el DataFrame oficial de Asset Allocation Dinámico
df_pesos_dinamico = pd.DataFrame(lista_pesos_dinamicos, index=fechas_rebalanceo, columns=activos)

# 4. Conectamos con el motor de simulación del Broker para auditar los costos reales
motor_broker = BrokerFactory.obtener_broker("inviu_con_asesor")
friccion = motor_broker.simular_costos_operativos(df_pesos_dinamico, capital_inicial=capital_inicial)

# 5. Métricas de control impresas en consola
print("\n====== AUDITORÍA DE FRICCIÓN FINANCIERA (INVIU) EN VALIDACIÓN ======")
print(f"Broker Simulado:            {friccion['label_broker']}")
print(f"Volumen Nominal Operado:    ${friccion['volumen_nominal_operado']:.2f}")
print(f"Aranceles Netos Devengados: ${friccion['aranceles_netos']:.2f}")
print(f"IVA Estructural (21%):      ${friccion['iva_sobre_comisiones']:.2f}")
print(f"Fricción Total Acumulada:   ${friccion['friccion_total_acumulada']:.2f}")
print(f"Veces que rotó el capital:  {friccion['veces_rotado_capital']:.2f}x")
print("====================================================================")

⏳ Ejecutando optimizaciones mensuales Walk-Forward en periodo de validación...

====== AUDITORÍA DE FRICCIÓN FINANCIERA (INVIU) EN VALIDACIÓN ======
Broker Simulado:            Inviu SA (Con Asesor - Arancel: 1.5% + IVA)
Volumen Nominal Operado:    $54938.88
Aranceles Netos Devengados: $824.08
IVA Estructural (21%):      $173.06
Fricción Total Acumulada:   $997.14
Veces que rotó el capital:  5.49x


# Procesamiento de KPIs de Riesgo y Retorno

In [5]:
# ==============================================================================
# ASIGNACIÓN MATRICIAL USANDO TUS VARIABLES LOGARÍTMICAS
# ==============================================================================
estrategias = {
    "Portafolio Equitativo (1/N)": ret_log_oos_1n,
    "Markowitz Estático Optimizado": ret_log_oos_estatico,
    "Markowitz Regularizado (Dinámico)": ret_log_oos_regularizado
}

# ==============================================================================
# BUCLE DE CÁLCULO DE ALFA (α) Y BETA (β) EN CONSOLA
# ==============================================================================
ret_bench_diarios = FinancialStats.calcular_retornos_log(curva_benchmark)

varianza_bench = np.var(ret_bench_diarios)
ret_anual_bench = ret_bench_diarios.mean() * 252

print(f"\n====================================================================")
print(f"📊 ANÁLISIS DE RIESGO SISTEMÁTICO VS BENCHMARK: {ticker_benchmark}")
print(f"====================================================================")

# Diccionario para guardar los resultados métricos
resultados_mercado = {}

for nombre, serie_retornos in estrategias.items():
    # Sincronizamos las series por fechas por seguridad (ej. feriados locales vs Wall Street)
    serie_sinc, bench_sinc = serie_retornos.align(ret_bench_diarios, join='inner', axis=0)
    
    # Cálculo del Beta (β) -> Covarianza(Estrategia, Bench) / Varianza(Bench)
    matriz_cov = np.cov(serie_sinc, bench_sinc)
    covarianza_cruzada = matriz_cov[0, 1]
    beta = covarianza_cruzada / varianza_bench
    
    # Cálculo del Alfa (α) Anualizado -> Retorno_Est - [Rf + Beta * (Ret_Bench - Rf)]
    ret_anual_estrategia = serie_sinc.mean() * 252
    alfa = ret_anual_estrategia - (tasa_libre_riesgo_anual + beta * (ret_anual_bench - tasa_libre_riesgo_anual))
    
    # Guardamos en memoria para usar después en el reporte
    resultados_mercado[nombre] = {"beta": beta, "alfa": alfa}
    
    # Imprimimos los resultados directamente en tu notebook
    print(f"\n📈 {nombre.upper()}:")
    print(f"   -> Beta (β): {beta:.2f} " + ("(Más volátil que el índice)" if beta > 1 else "(Más defensiva/estable que el índice)"))
    print(f"   -> Alfa (α) Anualizado: {alfa:.2%} " + ("(¡Generó Alfa positivo!) 🚀" if alfa > 0 else "(Performance inferior al riesgo tomado)"))

print(f"====================================================================")


📊 ANÁLISIS DE RIESGO SISTEMÁTICO VS BENCHMARK: QQQ

📈 PORTAFOLIO EQUITATIVO (1/N):
   -> Beta (β): 1.12 (Más volátil que el índice)
   -> Alfa (α) Anualizado: -0.61% (Performance inferior al riesgo tomado)

📈 MARKOWITZ ESTÁTICO OPTIMIZADO:
   -> Beta (β): 1.21 (Más volátil que el índice)
   -> Alfa (α) Anualizado: 4.78% (¡Generó Alfa positivo!) 🚀

📈 MARKOWITZ REGULARIZADO (DINÁMICO):
   -> Beta (β): 1.20 (Más volátil que el índice)
   -> Alfa (α) Anualizado: 3.53% (¡Generó Alfa positivo!) 🚀


In [6]:
estrategias_oos = [
    ("Benchmark", curva_benchmark),
    ("Portafolio Equitativo (1/N)", curva_oos_1n),
    ("Markowitz Estático Optimizado", curva_oos_estatico),
    ("Markowitz Regularizado (Dinámico)", curva_oos_regularizado)
]

tabla_kpis = []
for nombre, curva in estrategias_oos:
    anios_oos = len(curva) / 252
    ret_anualizado = (curva.iloc[-1] / curva.iloc[0]) - 1
    
    alfa_estrategia = resultados_mercado.get(nombre, {}).get("alfa", 0.0)
    # Invocación blindada de tus funciones estadísticas
    tabla_kpis.append([
        nombre,
        ret_anualizado,
        FinancialStats.get_anual_volatility(curva),
        FinancialStats.get_ratio_sharpe(curva, risk_free_rate=0.04),
        FinancialStats.get_ratio_sortino(curva, risk_free_rate=0.04),
        FinancialStats.get_max_drawdown(curva),
        FinancialStats.get_ratio_calmar(curva),
        alfa_estrategia
    ])

# Render de control en Jupyter
pd.DataFrame(tabla_kpis, columns=["ESTRATEGIA (VALIDACIÓN OOS)", "RETORNO", "VOLATILIDAD", "SHARPE", "SORTINO", "MAX DD", "CALMAR","ALPHA (α) ANUAL"])

,ESTRATEGIA (VALIDACIÓN OOS),RETORNO,VOLATILIDAD,SHARPE,SORTINO,MAX DD,CALMAR,ALPHA (α) ANUAL
0,Benchmark,0.200531,0.236689,0.735579,0.958136,-0.227683,0.984265,0.000000
1,Portafolio Equitativo (1/N),0.232470,0.277228,0.758353,1.088809,-0.256700,0.936914,-0.006057
2,Markowitz Estático Optimizado,0.320764,0.305147,0.945868,1.314711,-0.295864,1.110261,0.047777
3,Markowitz Regularizado (Dinámico),0.303861,0.302315,0.908593,1.265209,-0.288331,1.084513,0.035294


In [7]:
# ==============================================================================
# CÓMPUTO DE RENDIMIENTOS MENSUALES (PERIODO OUT-OF-SAMPLE)
# ==============================================================================

# Agrupamos los retornos logarítmicos diarios por mes/año y los sumamos
ret_log_mensual = pd.DataFrame({
    "Equitativo (1/N)": ret_log_oos_1n,
    "Markowitz Estático": ret_log_oos_estatico,
    "Markowitz Regularizado (Dinámico)": ret_log_oos_regularizado
}, index=retornos_oos.index)


ret_log_mensual.index = pd.to_datetime(ret_log_mensual.index)
# Resample por fin de mes ('ME') sumando retornos logarítmicos, luego convertimos a simple
rendimientos_mensuales = ret_log_mensual.resample('ME').sum().apply(lambda x: np.exp(x) - 1)

# Formateamos el índice para que sea legible (Ej: 2025-01)
rendimientos_mensuales.index = rendimientos_mensuales.index.to_period('M')

print("📊 Rendimientos Mensuales en Validación (OOS):")
rendimientos_mensuales.style.format("{:.2%}")

📊 Rendimientos Mensuales en Validación (OOS):


,Equitativo (1/N),Markowitz Estático,Markowitz Regularizado (Dinámico)
Fecha,,,
2025-01,2.21%,-3.69%,-3.52%
2025-02,-4.91%,-2.55%,-2.26%
2025-03,-10.04%,-10.00%,-9.92%
2025-04,-0.68%,-0.03%,0.24%
2025-05,11.70%,10.24%,11.50%
2025-06,8.36%,8.12%,8.69%
2025-07,6.84%,7.58%,7.72%
2025-08,1.31%,4.42%,3.15%
2025-09,4.59%,8.78%,7.96%


In [8]:
# ==============================================================================
# SEGUIMIENTO DEL ASSET ALLOCATION MENSUAL (PERIODO OUT-OF-SAMPLE)
# ==============================================================================

# Opción A: Si usás la simulación de rebalanceo dinámico de la Celda 5 (df_pesos_dinamico)
# Formateamos el índice temporal de las decisiones mensuales para la tabla
asset_allocation_mensual = df_pesos_dinamico.copy()
asset_allocation_mensual.index = pd.to_datetime(asset_allocation_mensual.index).to_period('M')

print("🥧 Asignación de Activos (Asset Allocation) Dinámica Mensual:")
asset_allocation_mensual.style.format("{:.2%}")

🥧 Asignación de Activos (Asset Allocation) Dinámica Mensual:


,AAPL,MSFT,GOOGL,AMZN,NVDA,META
2025-01,23.73%,17.06%,7.19%,34.65%,0.00%,17.38%
2025-02,35.00%,0.23%,9.92%,35.00%,0.00%,19.85%
2025-03,35.00%,2.29%,0.00%,35.00%,0.00%,27.71%
2025-04,35.00%,0.00%,0.00%,35.00%,0.00%,30.00%
2025-05,9.34%,22.39%,0.00%,35.00%,0.00%,33.27%
2025-06,0.00%,15.01%,0.00%,35.00%,16.72%,33.27%
2025-07,0.00%,0.00%,0.00%,35.00%,35.00%,30.00%
2025-08,0.00%,5.31%,33.04%,35.00%,17.68%,8.97%
2025-09,0.00%,0.00%,35.00%,16.51%,17.68%,30.81%
2025-10,0.00%,11.79%,35.00%,0.00%,32.46%,20.75%


In [9]:
# ==============================================================================
# CÓMPUTO DE PROFIT NETO FINAL (PERIODO OUT-OF-SAMPLE)
# ==============================================================================

# 1. Calculamos los Retornos Simples Brutos punta a punta directos de la curva
ret_simple_bruto_1n = (curva_oos_1n.iloc[-1] / curva_oos_1n.iloc[0]) - 1
ret_simple_bruto_estatico = (curva_oos_estatico.iloc[-1] / curva_oos_estatico.iloc[0]) - 1
ret_simple_bruto_dinamico = (curva_oos_regularizado.iloc[-1] / curva_oos_regularizado.iloc[0]) - 1
ret_simple_bruto_bench = (curva_benchmark.iloc[-1] / curva_benchmark.iloc[0]) - 1

# 2. Convertimos a Valor Final Bruto en dólares usando el retorno simple unificado
val_final_1n = capital_inicial * (1 + ret_simple_bruto_1n)
val_final_estatico = capital_inicial * (1 + ret_simple_bruto_estatico)
val_final_dinamico = capital_inicial * (1 + ret_simple_bruto_dinamico)
val_neto_bench = capital_inicial * (1 + ret_simple_bruto_bench)

# 3. Descontamos la fricción acumulada del broker únicamente a la estrategia que operó (Dinámica)
costo_total_friccion = friccion['friccion_total_acumulada']

val_neto_1n = val_final_1n
val_neto_estatico = val_final_estatico
val_neto_dinamico = val_final_dinamico - costo_total_friccion
val_neto_bench = val_neto_bench

# 4. Calculamos la Ganancia Neta Real (Profit Neto)
profit_neto_1n = val_neto_1n - capital_inicial
profit_neto_estatico = val_neto_estatico - capital_inicial
profit_neto_dinamico = val_neto_dinamico - capital_inicial
profit_neto_bench = val_neto_bench - capital_inicial


# 4. Estructuramos la lista para ReportLab
tabla_profit_neto = [
    ["Estrategia", "Monto Final Neto", "Ganancia Neta Absoluta (Profit)", "Retorno Neto Total"],
    ["Benchmark", f"${val_neto_bench:,.2f}", f"${profit_neto_bench:,.2f}", f"{(val_neto_bench/capital_inicial - 1) :.2%}"],
    ["Portafolio Equitativo (1/N)", f"${val_neto_1n:,.2f}", f"${profit_neto_1n:,.2f}", f"{(val_neto_1n/capital_inicial - 1):.2%}"],
    ["Markowitz Estático Optimizado", f"${val_neto_estatico:,.2f}", f"${profit_neto_estatico:,.2f}", f"{(val_neto_estatico/capital_inicial - 1):.2%}"],
    ["Markowitz Regularizado (Dinámico)", f"${val_neto_dinamico:,.2f}", f"${profit_neto_dinamico:,.2f}", f"{(val_neto_dinamico/capital_inicial - 1):.2%}"]
]

# Construccion Tabla de Fricción

In [10]:

# Transformamos las curvas a valores monetarios nominales para el reporte visual
df_equity_plots = pd.DataFrame({
    "Portafolio Equitativo": curva_oos_1n * capital_inicial,
    "Markowitz Estático": curva_oos_estatico * capital_inicial,
    "Markowitz Dinámico (Inviu)": curva_oos_regularizado * capital_inicial
}, index=retornos_oos.index)

# Extraemos la distribución final del entrenamiento regularizado para el gráfico de torta
pesos_actuales_serie = pd.Series(pesos_markowitz_regularizado, index=activos)

# Generación de vectores gráficos (.png)
path_line, path_pie = QuantPlotter.generar_graficos_reporte(df_equity_plots, pesos_actuales_serie)

anios_oos = len(curva_oos_regularizado) / 252
ret_anual_regularizado = (curva_oos_regularizado.iloc[-1]) ** (1 / anios_oos) - 1
ganancia_bruta_teorica = ret_anual_regularizado * capital_inicial

# 2. Calculamos el porcentaje real devorado por el broker en este periodo OOS
porcentaje_devorado = (friccion['friccion_total_acumulada'] / ganancia_bruta_teorica) * 100
# Construcción de tablas de fricción auxiliares
tabla_friccion = [
    ["Volumen Nominal Operado", f"${friccion['volumen_nominal_operado']:.2f}", f"Estrategia rotó {friccion['veces_rotado_capital']:.1f} veces el capital inicial bruto.", False],
    ["Aranceles Netos Inviu", f"${friccion['aranceles_netos']:.2f}", "Pérdida directa por fee neto de intermediación.", False],
    ["IVA Estructural (21%)", f"${friccion['iva_sobre_comisiones']:.2f}", "Costo impositivo estructural no recuperable en mercado local.", False],
    ["Fricción Total Acumulada", f"${friccion['friccion_total_acumulada']:.2f}", f"Se devoró un {porcentaje_devorado:.2f}% del rendimiento bruto teórico en este periodo.", True]
]



# Construccion de Rendimientos mensuales y Asset Allocation

In [11]:
tabla_rendimientos_mensuales = [["Mes/Año", "Equitativo (1/N)", "Markowitz Estático", "Markowitz Regularizado (Dinámico)"]]
for idx, row in rendimientos_mensuales.iterrows():
    tabla_rendimientos_mensuales.append([
        str(idx),
        f"{row['Equitativo (1/N)']:.2%}",
        f"{row['Markowitz Estático']:.2%}",
        f"{row['Markowitz Regularizado (Dinámico)']:.2%}"
    ])

# Tabla B: Asset Allocation Mensual (Mapeamos la rotación dinámica de activos)
columnas_activos = list(asset_allocation_mensual.columns)
tabla_asset_allocation = [["Mes/Año"] + columnas_activos]
for idx, row in asset_allocation_mensual.iterrows():
    fila_formateada = [str(idx)] + [f"{row[activo]:.1%}" for activo in columnas_activos]
    tabla_asset_allocation.append(fila_formateada)

# Reporte de Análisis de Operaciones de Estrategias

In [12]:
# Extraemos dinámicamente las fechas exactas para evitar hardcoding
fecha_inicio_is = retornos_is.index.min().strftime('%m/%Y')
fecha_fin_is = retornos_is.index.max().strftime('%m/%Y')
fecha_inicio_oos = retornos_oos.index.min().strftime('%m/%Y')
fecha_fin_oos = retornos_oos.index.max().strftime('%m/%Y')

# Compilación final del documento vía ReportLab
output_pdf = "reporte_performance_validacion.pdf"
metadata = {
    "analista": "Juan Martin Rodriguez",
    "label_broker": friccion['label_broker'],
    "capital_inicial_prueba": f"${capital_inicial:,.2f}",
    "fase_entrenamiento_is": f"{fecha_inicio_is} a {fecha_fin_is} (In-Sample)",
    "fase_validacion_oos": f"{fecha_inicio_oos} a {fecha_fin_oos} (Out-of-Sample)",
    "descripcion_metodologica": (
        f"Optimización de pesos realizada exclusivamente con datos históricos del período IS. "
        f"Evaluación de performance y métricas de fricción operativa ejecutadas sobre entorno ciego (OOS) "
        f"con un capital inicial de ${capital_inicial:,.2f}."
    )
}

def evaluar_perfil(datos):
        beta = datos["beta"]
        alfa = datos["alfa"]
            
        # Matriz de decisión lógica
        if beta > 1.15 and alfa > 0:
            return "Agresivo / Extractor de Alfa"
        elif beta > 1.15 and alfa <= 0:
            return "Alta Volatilidad / Ineficiente"
        elif 0.85 <= beta <= 1.15 and alfa > 0:
            return "Riesgo Mercado / Eficiente"
        elif beta < 0.85 and alfa > 0:
            return "Defensivo / Cobertura Alfa"
        else:
            return "Riesgo Moderado / Indexado"

        # Generamos las descripciones dinámicas al vuelo
perfil_1n = evaluar_perfil(resultados_mercado['Portafolio Equitativo (1/N)'])
perfil_estatico = evaluar_perfil(resultados_mercado['Markowitz Estático Optimizado'])
perfil_dinamico = evaluar_perfil(resultados_mercado['Markowitz Regularizado (Dinámico)'])

tabla_mercado_data = [
            ["Estrategia", f"Beta (β) vs {ticker_benchmark}", "Alfa (α) Anualizado", "Perfil de Riesgo CAPM"],
            ["Portafolio Equitativo (1/N)", f"{resultados_mercado['Portafolio Equitativo (1/N)']['beta']:.2f}", f"{resultados_mercado['Portafolio Equitativo (1/N)']['alfa']:.2%}", perfil_1n],
            ["Markowitz Estático Optimizado", f"{resultados_mercado['Markowitz Estático Optimizado']['beta']:.2f}", f"{resultados_mercado['Markowitz Estático Optimizado']['alfa']:.2%}", perfil_estatico],
            ["Markowitz Regularizado (Dinámico)", f"{resultados_mercado['Markowitz Regularizado (Dinámico)']['beta']:.2f}", f"{resultados_mercado['Markowitz Regularizado (Dinámico)']['alfa']:.2%}", perfil_dinamico]
        ]

reporte = ReportGenerator(output_pdf)
reporte.construir_reporte(metadata, 
                          tabla_kpis, 
                          tabla_friccion, 
                          path_line, 
                          path_pie,
                          tabla_rendimientos_mensuales,
                          tabla_asset_allocation,
                          tabla_profit_neto,
                          ticker_benchmark,
                          tabla_mercado_data
)
reporte.generar()

print(f"🎉 ¡Pipeline completado! El informe '{output_pdf}' fue emitido utilizando métricas robustas Out-of-Sample.")

🎉 ¡Pipeline completado! El informe 'reporte_performance_validacion.pdf' fue emitido utilizando métricas robustas Out-of-Sample.
